In [2]:
import re , json , math , pickle , os , time
import pandas as pd 
import numpy as np
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from rapidfuzz import process , fuzz  # for fuzzy matching

df = pd.read_csv('cleaned_data/bangalore_cleaned.csv')
df.shape , df.columns

((7144, 9),
 Index(['area_type', 'availability', 'location', 'size', 'society',
        'total_sqft', 'bath', 'balcony', 'price'],
       dtype='object'))

In [3]:
# cell 2
def convert_sqft(x):
    try:
        s = str(x).strip()
        if "-" in s:
            a,b = s.split("-")
            return (float(a)+float(b))/2
        return float(s)
    except:
        return np.nan

def extract_bhk(x):
    if pd.isna(x): return np.nan
    m = re.search(r'(\d+)\s*(bhk|bedroom)', str(x).lower())
    if m: return int(m.group(1))
    m = re.search(r'(\d+)', str(x))
    if m: return int(m.group(1))
    return np.nan

# apply
df['total_sqft'] = df['total_sqft'].apply(convert_sqft)
df['bhk'] = df['size'].apply(extract_bhk)
# price column sometimes named 'price' or 'price_lakh' — convert to numeric (assume lakhs)
if 'price' in df.columns:
    df['price_lakh'] = pd.to_numeric(df['price'], errors='coerce')
elif 'price_lakh' in df.columns:
    df['price_lakh'] = pd.to_numeric(df['price_lakh'], errors='coerce')
else:
    raise Exception("No price column found")

# drop rows missing essential numeric fields
df = df.dropna(subset=['location','total_sqft','price_lakh','bhk'])
df = df.reset_index(drop=True)
df.shape

(7129, 11)

In [4]:
# cell 3
def row_to_text(r):
    parts = []
    parts.append(f"{int(r['bhk'])} BHK")
    parts.append(f"in {r['location']}")
    parts.append(f"{int(r['total_sqft'])} sqft")
    parts.append(f"priced at {float(r['price_lakh']):.2f} lakh")
    if 'bath' in r and not pd.isna(r['bath']):
        parts.append(f"{int(r['bath'])} bath(s)")
    if 'balcony' in r and not pd.isna(r['balcony']):
        parts.append(f"{int(r['balcony'])} balcony(ies)")
    if 'area_type' in r and not pd.isna(r['area_type']):
        parts.append(f"area_type: {r['area_type']}")
    if 'availability' in r and not pd.isna(r['availability']):
        parts.append(f"availability: {r['availability']}")
    if 'society' in r and not pd.isna(r['society']):
        parts.append(f"society: {r['society']}")
    return ". ".join(parts)

df['text'] = df.apply(row_to_text, axis=1)
# quick check
df[['location','total_sqft','price_lakh','bhk','text']].head()

,location,total_sqft,price_lakh,bhk,text
0,Electronic City Phase II,1056.0,39.07,2,2 BHK. in Electronic City Phase II. 1056 sqft....
1,Chikka Tirupathi,2600.0,120.00,4,4 BHK. in Chikka Tirupathi. 2600 sqft. priced ...
2,Lingadheeranahalli,1521.0,95.00,3,3 BHK. in Lingadheeranahalli. 1521 sqft. price...
3,Whitefield,1170.0,38.00,2,2 BHK. in Whitefield. 1170 sqft. priced at 38....
4,Whitefield,2785.0,295.00,4,4 BHK. in Whitefield. 2785 sqft. priced at 295...


In [6]:
# cell 4
llm = pipeline("text-generation", model="google/flan-t5-small")  # small & fast

LLM_PROMPT = """Extract filters from this user query and return EXACTLY a JSON object only.
Fields (use null for missing): location (string), min_sqft (number), max_price_lakh (number), bhk (int), min_bath (int).
Return example: {"location":"Whitefield","min_sqft":1500,"max_price_lakh":80,"bhk":3,"min_bath":2}
User query: "{q}"
"""

import json, re
from rapidfuzz import process, fuzz

def llm_extract_filters(query, sample_locations):
    prompt = LLM_PROMPT.format(q=query.replace('"','\\"'))
    out = llm(prompt, max_new_tokens=64, do_sample=False)[0]['generated_text']  # use only max_new_tokens
    # try to parse JSON substring
    j = None
    try:
        j = json.loads(out.strip())
    except:
        m = re.search(r"\{.*\}", out, flags=re.S)
        if m:
            try:
                j = json.loads(m.group(0))
            except:
                j = None
    return j

def regex_extract_filters(query):
    q = query.lower()
    filters={}
    m = re.search(r'under\s*([\d\.]+)\s*(lakh|lakhs|crore|crores|cr)?', q)
    if m:
        val = float(m.group(1))
        unit = m.group(2)
        if unit and 'crore' in unit: val *= 100
        filters['max_price_lakh'] = val
    m = re.search(r'(\d+)\s*(sqft|sq\.ft|square feet|sq ft)', q)
    if m:
        filters['min_sqft'] = float(m.group(1))
    m = re.search(r'(\d+)\s*(bhk|bedroom|bedrooms)', q)
    if m:
        filters['bhk'] = int(m.group(1))
    # location simple substring match
    for loc in sample_locations:
        if loc.lower() in q:
            filters['location'] = loc
            break
    m = re.search(r'(\d+)\s*(bath|baths|bathroom|bathrooms)', q)
    if m:
        filters['min_bath'] = int(m.group(1))
    return filters

def normalize_filters(raw, sample_locations):
    # start with empty normalized
    norm = {'location':None,'min_sqft':None,'max_price_lakh':None,'bhk':None,'min_bath':None}
    if not raw: return norm
    # helpers
    def to_num(x):
        if x is None: return None
        try: return float(x)
        except:
            s=re.sub(r'[^\d.]','',str(x))
            return float(s) if s!='' else None
    # copy possible keys
    for k in ['min_sqft','max_price_lakh','bhk','min_bath']:
        if k in raw:
            v = to_num(raw[k])
            if v is not None: 
                norm[k] = int(v) if k in ['bhk','min_bath'] else float(v)
    # location fuzzy match
    loc_candidate = raw.get('location') or raw.get('loc') or raw.get('city')
    if loc_candidate:
        best,score,_ = process.extractOne(str(loc_candidate), sample_locations, scorer=fuzz.WRatio)
        if score >= 65:
            norm['location'] = best
    return norm

def extract_user_filters(query, df):
    sample_locations = df['location'].dropna().unique().tolist()
    raw = llm_extract_filters(query, sample_locations)
    norm = normalize_filters(raw, sample_locations) if raw else {}
    # fallback to regex if nothing found
    if all(v is None for v in norm.values()):
        reg = regex_extract_filters(query)
        norm = normalize_filters(reg, sample_locations)
    return norm

# quick test
print(extract_user_filters("Show me 3 BHK properties in Whitefield under 80 lakh and above 1000 sqft", df))

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 633.23it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cp

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 633.23it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cp

KeyError: '"location"'